# 02 · 핵심 분석: 교차표·집단비교·회귀  (모듈 3)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/02_core_analysis.ipynb)

> 대표 분석 3종을 Python으로 수행하고, **같은 분석을 STATA로도**(각 절 🔁) 해서
> 두 결과의 숫자가 같은지 **교차검증**한다. — "숫자는 같다. 출력 모양만 다르다."

In [ ]:
import pandas as pd, numpy as np
from scipy import stats                       # t검정·ANOVA 등 통계 검정 함수
import statsmodels.formula.api as smf         # 회귀분석(R 스타일 수식 사용)
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv")                      # 데이터 불러오기
df["log_gdp"] = np.log(df["gdp_pc"])           # 로그변환 변수 미리 생성
df["log_pop"] = np.log(df["pop"])
print("준비 완료:", df.shape)

## 1) 교차표 — 지역 × 소득그룹 (국가 수)
지역·소득은 국가 속성이므로 **국가 단위로** 집계한다(연도 중복 제거).

In [ ]:
# 지역·소득은 '국가'의 속성 → 국가당 1행만 남겨 연도 중복 제거
countries = df.drop_duplicates("economy")
# 행=지역, 열=소득그룹 으로 국가 수를 센 교차표
pd.crosstab(countries["region_name"], countries["income_name"])

🔁 **STATA**: `tabulate region_name income_name`  ·  코드 → `stata/02_crosstab.do`

## 2) 집단 비교 — 지역 격차(t검정) · 소득그룹 차이(ANOVA)
- **t검정** = *두* 집단의 평균이 다른지, **ANOVA** = *셋 이상* 집단의 평균이 다른지 검정.
- 둘 다 **`p` < 0.05면 "우연으로 보기 어렵다(통계적으로 유의)"**.

In [ ]:
# t검정: 두 집단(사하라이남 아프리카 vs 그 외)의 기대수명 평균 차이
ssa  = df.loc[df.region_name == "Sub-Saharan Africa", "life_exp"].dropna()   # 사하라이남 값만
rest = df.loc[df.region_name != "Sub-Saharan Africa", "life_exp"].dropna()   # 나머지 값만
t, p = stats.ttest_ind(ssa, rest, equal_var=False)   # 분산이 달라 Welch 검정(STATA는 ttest ..., unequal)
print(f"[t검정] 사하라이남={ssa.mean():.1f}세  그외={rest.mean():.1f}세  t={t:.1f}  p={p:.1e}")  # p 작으면 차이 유의

# ANOVA: 셋 이상 집단(소득그룹 4개) 간 기대수명 평균 차이
groups = [g["life_exp"].dropna().values for _, g in df.groupby("income_name")]  # 소득그룹별 값 묶음
F, p2 = stats.f_oneway(*[g for g in groups if len(g) > 1])   # 일원분산분석(F가 클수록 차이 큼)
print(f"[ANOVA] 소득그룹별 기대수명  F={F:.0f}  p={p2:.1e}")

🔁 **STATA**: `ttest life_exp, by(ssa) unequal`  ·  `oneway life_exp income_n`  ·  코드 → `stata/03_group_compare.do`

## 3) 회귀분석 — Preston 곡선 (소득 → 기대수명)
**회귀** = 한 변수(기대수명)를 다른 변수(소득)로 설명하는 직선/곡선을 찾는 것. 1인당 GDP는 왜도가
크므로 **로그**를 취한다 — AI가 자동으로 안 할 수 있는 **설계 판단**.

> **출력 읽는 법**: `coef`=기울기(log소득 1↑당 수명 변화), `P>|t|`<0.05면 유의, `R²`=설명력(0~1).

In [ ]:
# "life_exp ~ log_gdp" = 기대수명을 log소득으로 설명. ols=최소제곱법, .fit()=계수 추정
m = smf.ols("life_exp ~ log_gdp", data=df).fit()
print(m.summary().tables[1])      # 계수표: coef(기울기)·std err(오차)·P>|t|(유의확률)
print(f"\nR² = {m.rsquared:.3f}")  # 설명력(0~1): 소득이 기대수명 변동의 몇 %를 설명하나

**해석**: `log_gdp` 계수가 **양수** → 소득이 높을수록 기대수명↑. (소득이 e배면 ≈ +계수 만큼 수명)
R²가 높아 소득만으로 기대수명 차이의 상당 부분이 설명된다.

🔁 **STATA**: `regress life_exp log_gdp` — **한 줄**.  코드 → `stata/04_regression.do`
> 양쪽 계수·R²가 같은지 **교차검증**: 같으면 신뢰, 다르면 하나가 틀린 것.

---
✅ **핵심**: 교차표·검정·회귀를 입문자가 첫날 직접 돌렸다. 비결은 *문법 암기*가 아니라
*무엇을·왜 분석할지 설계*하고 *검증·해석*하는 것.